### Learning Retrieval of RAG

In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

c:\Users\Rupesh\Desktop\learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Rupesh\AppData\Local\Temp\ipykernel_22488\1363179362.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("Please set the GROQ_API_KEY environment variable.")

model_name = "llama-3.3-70b-versatile"   # or another Groq-supported model
temperature = 0.0

llm = ChatGroq(
    model=model_name,
    temperature=temperature,
    groq_api_key=groq_api_key
)


In [3]:
loader = TextLoader("./work.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(documents)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")  # ← changed


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1538.60it/s]


In [4]:
print(len(texts), "chunks created")

4 chunks created


In [5]:
database = FAISS.from_documents(texts, embeddings)
retriever = database.as_retriever(search_kwargs={"k": 3})


def query_database(query):
    results = retriever.invoke(query)   # ← use invoke() not get_relevant_documents()
    if not results:
        return "No relevant documents found."
    return results

In [6]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F4FE777650>, search_kwargs={'k': 3})

In [7]:
relevant_documents = query_database("What is the main topic of the document?")

In [8]:
for doc in relevant_documents:
    print(f"Document: {doc.page_content[:200]}...")  # Print the first 200 characters of each document
    print(f"Metadata: {doc.metadata}\n")  # Print metadata if available
    print("-" * 80)  # Separator for clarity

Document: "Didn't it get boring when you got to be about 15?" I asked.

"No," he said, "by then I was interested in maths."

In another conversation he told me that what he really liked was solving problems. To...
Metadata: {'source': './work.txt'}

--------------------------------------------------------------------------------
Document: It seemed curious that the same task could be painful to one person and pleasant to another, but I didn't realize at the time what this imbalance implied, because I wasn't looking for it. I didn't rea...
Metadata: {'source': './work.txt'}

--------------------------------------------------------------------------------
Document: Few people know so early or so certainly what they want to work on. But talking to my father reminded me of a heuristic the rest of us can use. If something that seems like work to other people doesn'...
Metadata: {'source': './work.txt'}

--------------------------------------------------------------------------------


In [9]:
relevant_documents = query_database(
    "What types of things did the author want to build?")

print("\n\n".join(
    [
        f"Document: {doc.page_content[:200]}...\nMetadata: {doc.metadata}"
        for doc in relevant_documents
    ]
))

Document: "Didn't it get boring when you got to be about 15?" I asked.

"No," he said, "by then I was interested in maths."

In another conversation he told me that what he really liked was solving problems. To...
Metadata: {'source': './work.txt'}

Document: What Doesn't Seem Like Work?

January 2015

My father is a mathematician. For most of my childhood he worked for Westinghouse, modelling nuclear reactors.

He was one of those lucky people who know ea...
Metadata: {'source': './work.txt'}

Document: It seemed curious that the same task could be painful to one person and pleasant to another, but I didn't realize at the time what this imbalance implied, because I wasn't looking for it. I didn't rea...
Metadata: {'source': './work.txt'}
